In [1]:
# =========================================================
# STEP 2 — DATA CLEANING & PREPROCESSING
# FILE: notebooks/01_data_cleaning.ipynb
# PROJECT: Smart Complaint Analytics System
# INPUT : data/raw/government_complaints.csv
# OUTPUT: data/processed/clean_complaints.csv
# =========================================================

# =========================================================
# 2.1 IMPORT LIBRARIES
# =========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# =========================================================
# 2.2 LOAD DATASET
# =========================================================

DATA_PATH = Path("../data/raw/government_complaints.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["created_at"])

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 16)


,complaint_id,created_at,channel,citizen_comment,complaint_category,department,district,village,sentiment,urgency_level,priority_score,status,response_time_hours,resolution_time_days,sla_met,engagement_score
0,CMP-04708,2024-01-01 03:40:36,Call Center,Tempat pembuangan di Dadaprejo sudah penuh.,Waste Management,Dinas Lingkungan Hidup,Junrejo,Dadaprejo,Negative,Medium,63,Resolved,30.33,3.23,No,3877
1,CMP-07774,2024-01-01 03:43:25,Call Center,Aspal rusak parah di daerah Temas dan belum di...,Road Infrastructure,Dinas PUPR,Batu,Temas,Negative,Low,37,Resolved,101.39,10.42,No,845
2,CMP-01189,2024-01-01 06:35:30,LAPOR!,Air PDAM di Mojorejo keruh dan tidak layak dig...,Clean Water,PDAM,Bumiaji,Mojorejo,Neutral,Medium,42,Open,24.42,NaN,No,1187
3,CMP-01853,2024-01-01 10:09:18,Website,Genangan air sering terjadi di Dadaprejo saat ...,Flooding,Dinas PUPR,Junrejo,Dadaprejo,Neutral,High,80,In Progress,16.67,NaN,No,4440
4,CMP-07824,2024-01-01 14:43:25,TikTok,Lampu jalan rusak di kawasan Gunungsari.,Street Lighting,Dinas Perhubungan,Junrejo,Gunungsari,Negative,Medium,44,Resolved,11.35,8.32,Yes,1780


In [2]:
# =========================================================
# 2.3 INITIAL DATA AUDIT
# =========================================================

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

DATASET OVERVIEW

Shape:
(10000, 16)

Data Types:
complaint_id                    object
created_at              datetime64[ns]
channel                         object
citizen_comment                 object
complaint_category              object
department                      object
district                        object
village                         object
sentiment                       object
urgency_level                   object
priority_score                   int64
status                          object
response_time_hours            float64
resolution_time_days           float64
sla_met                         object
engagement_score                 int64
dtype: object

Missing Values:
complaint_id               0
created_at                 0
channel                    0
citizen_comment            0
complaint_category         0
department                 0
district                   0
village                    0
sentiment                  0
urgency_level              0
prior

In [3]:
# =========================================================
# 2.4 REMOVE DUPLICATES
# =========================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Rows Before : {before}")
print(f"Rows After  : {after}")
print(f"Removed     : {before - after}")

Rows Before : 10000
Rows After  : 10000
Removed     : 0


In [4]:
# =========================================================
# 2.5 STANDARDIZE TEXT COLUMNS
# =========================================================

text_columns = [
    "channel",
    "citizen_comment",
    "complaint_category",
    "department",
    "district",
    "village",
    "sentiment",
    "urgency_level",
    "status",
    "sla_met"
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text standardization completed.")

Text standardization completed.


In [5]:
# =========================================================
# 2.6 HANDLE MISSING VALUES
# =========================================================

# Resolution time hanya kosong jika status belum Resolved.
# Biarkan NaN karena secara bisnis memang belum selesai.

numeric_fill_zero = [
    "engagement_score",
    "priority_score"
]

for col in numeric_fill_zero:
    df[col] = df[col].fillna(0)

print("Missing value handling completed.")

Missing value handling completed.


In [6]:
# =========================================================
# 2.7 FEATURE ENGINEERING
# =========================================================

# Date-based features
df["year"] = df["created_at"].dt.year
df["month"] = df["created_at"].dt.month
df["month_name"] = df["created_at"].dt.strftime("%B")
df["quarter"] = df["created_at"].dt.quarter
df["day_of_week"] = df["created_at"].dt.day_name()

# Operational features
df["is_resolved"] = np.where(df["status"] == "Resolved", 1, 0)
df["sla_met_flag"] = np.where(df["sla_met"] == "Yes", 1, 0)
df["is_critical"] = np.where(df["urgency_level"] == "Critical", 1, 0)

# Priority classification
def priority_bucket(score):
    if score >= 90:
        return "Critical"
    elif score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

df["priority_bucket"] = df["priority_score"].apply(priority_bucket)

print("Feature engineering completed.")

Feature engineering completed.


In [7]:
# =========================================================
# 2.8 BUSINESS KPI PREVIEW
# =========================================================

resolution_rate = df["is_resolved"].mean() * 100
sla_compliance = df["sla_met_flag"].mean() * 100
critical_rate = df["is_critical"].mean() * 100
avg_response_time = df["response_time_hours"].mean()

print("=" * 60)
print("EXECUTIVE KPI PREVIEW")
print("=" * 60)
print(f"Resolution Rate      : {resolution_rate:.2f}%")
print(f"SLA Compliance       : {sla_compliance:.2f}%")
print(f"Critical Complaint   : {critical_rate:.2f}%")
print(f"Avg Response Time    : {avg_response_time:.2f} hours")

EXECUTIVE KPI PREVIEW
Resolution Rate      : 47.79%
SLA Compliance       : 38.68%
Critical Complaint   : 8.01%
Avg Response Time    : 35.73 hours


In [8]:
# =========================================================
# 2.9 DATA QUALITY VALIDATION
# =========================================================

print("\nUnique Complaint Categories:")
print(df["complaint_category"].nunique())

print("\nUnique Departments:")
print(df["department"].nunique())

print("\nUnique Channels:")
print(df["channel"].nunique())

print("\nUnique Urgency Levels:")
print(df["urgency_level"].unique())

print("\nUnique Status Values:")
print(df["status"].unique())


Unique Complaint Categories:
10

Unique Departments:
8

Unique Channels:
6

Unique Urgency Levels:
['Medium' 'Low' 'High' 'Critical']

Unique Status Values:
['Resolved' 'Open' 'In Progress']


In [9]:
# =========================================================
# 2.10 SAVE CLEAN DATASET
# =========================================================

OUTPUT_PATH = Path("../data/processed/clean_complaints.csv")

# Pastikan folder ada
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Clean dataset saved to: {OUTPUT_PATH}")
print("Final Shape:", df.shape)

Clean dataset saved to: ..\data\processed\clean_complaints.csv
Final Shape: (10000, 25)


In [10]:
# =========================================================
# 2.11 FINAL EXECUTIVE SUMMARY
# =========================================================

print("""
============================================================
SMART COMPLAINT ANALYTICS SYSTEM
DATA CLEANING EXECUTIVE SUMMARY
============================================================

✔ Duplicate records removed
✔ Missing values handled
✔ Text columns standardized
✔ Date-based features engineered
✔ Operational KPI features created
✔ Clean dataset exported successfully

Output File:
data/processed/clean_complaints.csv

The dataset is now ready for:
1. Complaint Intelligence Analytics (EDA)
2. Complaint Category Classification
3. Urgency Detection
4. SLA Performance Analysis
5. Executive Dashboard Development
============================================================
""")


SMART COMPLAINT ANALYTICS SYSTEM
DATA CLEANING EXECUTIVE SUMMARY

✔ Duplicate records removed
✔ Missing values handled
✔ Text columns standardized
✔ Date-based features engineered
✔ Operational KPI features created
✔ Clean dataset exported successfully

Output File:
data/processed/clean_complaints.csv

The dataset is now ready for:
1. Complaint Intelligence Analytics (EDA)
2. Complaint Category Classification
3. Urgency Detection
4. SLA Performance Analysis
5. Executive Dashboard Development

